In [ ]:
# ===================================================================
# Video instance segmentation on DAVIS 2017 val.
#
# Mask R-CNN (COCO-pretrained, no fine-tuning) is run frame by frame.
# DAVIS object identities are class-agnostic, so predicted masks are
# matched to GT instances by mask IoU without regard to COCO class.
#
# Two kinds of number are reported:
#   1. per-frame quality  - mean IoU (J) and class-agnostic mask AP
#   2. TEMPORAL behaviour - how often the prediction covering the same
#      GT object jumps to a different predicted identity, how often it
#      flickers in and out, and how much less stable the predicted mask
#      is frame to frame than the GT mask of the same object. That last
#      comparison is what separates model instability from object motion.
#
# Masks are held for one sequence at a time; only compact RLEs are kept
# across sequences, because 480p bool masks for the whole split would
# be several GB.
# ===================================================================
import os, sys, json, glob, time, collections
import numpy as np
print("python", sys.version)
import torch, torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__, "cuda", torch.cuda.is_available())
from PIL import Image

ROOT = "/kaggle/input"
print("TOP LEVEL:", sorted(os.listdir(ROOT)))

ext = collections.Counter(); bydir = collections.Counter(); samples = []
ndirs = 0
for dp, dn, fn in os.walk(ROOT):          # NO depth cap
    ndirs += 1
    if fn:
        bydir[os.path.relpath(dp, ROOT)] += len(fn)
    for f in fn:
        ext[os.path.splitext(f)[1].lower()] += 1
        if len(samples) < 25:
            samples.append(os.path.join(os.path.relpath(dp, ROOT), f))
print("dirs walked:", ndirs)
print("extensions:", ext.most_common(10))
print("sample real paths:")
for s in samples:
    print("   ", s)
print("dirs containing ImageSets or ending in 480p:")
for d in sorted(bydir):
    if "ImageSets" in d or d.endswith("480p"):
        print("  %8d %s" % (bydir[d], d))

In [ ]:
# ---- locate DAVIS ----
ann_roots = [p for p in glob.glob(ROOT + "/**/Annotations/480p", recursive=True) if os.path.isdir(p)]
img_roots = [p for p in glob.glob(ROOT + "/**/JPEGImages/480p", recursive=True) if os.path.isdir(p)]
print("Annotations/480p:", ann_roots)
print("JPEGImages/480p:", img_roots)
assert ann_roots and img_roots, "DAVIS 480p Annotations/JPEGImages not found"
ANN_ROOT, IMG_ROOT = ann_roots[0], img_roots[0]

val_lists = glob.glob(ROOT + "/**/ImageSets/2017/val.txt", recursive=True)
print("val.txt candidates:", val_lists)
if val_lists:
    SEQS = [l.strip() for l in open(val_lists[0]) if l.strip()]
    SPLIT_SRC = "official DAVIS 2017 ImageSets/2017/val.txt"
else:
    SEQS = sorted(os.listdir(ANN_ROOT))
    SPLIT_SRC = "all sequences present (no official val.txt in this dataset copy)"
SEQS = [s for s in SEQS if os.path.isdir(os.path.join(ANN_ROOT, s)) and os.path.isdir(os.path.join(IMG_ROOT, s))]
print("split source:", SPLIT_SRC)
print("sequences:", len(SEQS), SEQS)
assert len(SEQS) >= 10, "only %d sequences usable" % len(SEQS)

tot_frames = sum(len([f for f in os.listdir(os.path.join(ANN_ROOT, s))
                      if os.path.basename(f).lower().endswith(".png")]) for s in SEQS)
print("annotated frames in split:", tot_frames)
assert tot_frames >= 500, "only %d annotated frames" % tot_frames

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
import torchvision.transforms.functional as TFUNC
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as maskutil

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "GPU not attached"
WT = MaskRCNN_ResNet50_FPN_Weights.COCO_V1
model = maskrcnn_resnet50_fpn(weights=WT).eval().to(dev)
print("weights:", WT, "| published COCO metrics:", WT.meta.get("_metrics"))
CATS = WT.meta.get("categories", [])

SCORE_TH = 0.5
MASK_TH = 0.5
LINK_TH = 0.5     # IoU to call two masks in consecutive frames the same predicted object
DET_TH = 0.5      # IoU to call a GT object detected in a frame

def gt_instances(path):
    a = np.array(Image.open(path))
    if a.ndim == 3:
        a = a[:, :, 0]
    out = {}
    for v in np.unique(a):
        if v == 0 or v == 255:
            continue
        out[int(v)] = (a == v)
    return out

def iou(a, b):
    inter = np.logical_and(a, b).sum()
    if inter == 0:
        return 0.0
    return float(inter) / float(np.logical_or(a, b).sum())

def rle(m):
    r = maskutil.encode(np.asfortranarray(m.astype(np.uint8)))
    r["counts"] = r["counts"].decode("ascii")
    return r

In [ ]:
# ---- one pass, one sequence at a time ----
images, annos, dets = [], [], []
aid = 1
imid = 1
per_seq = {}
per_seq_temporal = {}
all_ious = []
jumps = pairs_checked = flicker = flicker_pairs = 0
pred_stab, gt_stab = [], []
lab_hits = collections.Counter()
n_scored_frames = n_inst_frames = n_pred_masks = 0
COVER_TH = 0.5            # a GT object counts as covered if its mean best-IoU over its frames reaches this
OBJ_MEAN_IOU, OBJ_COVERED, OBJ_KEY = [], [], []
t0 = time.time()

for si, seq in enumerate(SEQS):
    apngs = sorted(f for f in os.listdir(os.path.join(ANN_ROOT, seq))
                   if os.path.basename(f).lower().endswith(".png"))
    GL, PL, FL = [], [], []
    for f in apngs:
        stem = os.path.splitext(f)[0]
        jpg = os.path.join(IMG_ROOT, seq, stem + ".jpg")
        if not os.path.exists(jpg):
            continue
        g = gt_instances(os.path.join(ANN_ROOT, seq, f))
        if not g:
            continue
        im = Image.open(jpg).convert("RGB")
        with torch.inference_mode():
            o = model([TFUNC.to_tensor(im).to(dev)])[0]
        keep = o["scores"] > SCORE_TH
        m = (o["masks"][keep, 0].cpu().numpy() > MASK_TH)
        sc = o["scores"][keep].cpu().numpy()
        lb = o["labels"][keep].cpu().numpy()
        PL.append([(m[i], float(sc[i]), int(lb[i])) for i in range(len(sc))])
        GL.append(g)
        FL.append(stem)
    n_scored_frames += len(GL)
    n_inst_frames += sum(len(g) for g in GL)
    n_pred_masks += sum(len(p) for p in PL)

    # --- per-frame quality + COCO records ---
    seq_iou = collections.defaultdict(list)
    for fi, (gl, pl) in enumerate(zip(GL, PL)):
        h, w = next(iter(gl.values())).shape
        images.append({"id": imid, "file_name": "%s/%s" % (seq, FL[fi]),
                       "height": int(h), "width": int(w)})
        for gid, gm in gl.items():
            best, bl = 0.0, None
            for pm, sc, lb in pl:
                v = iou(gm, pm)
                if v > best:
                    best, bl = v, lb
            seq_iou[gid].append(best)
            all_ious.append(best)
            if best >= 0.5 and bl is not None and bl < len(CATS):
                lab_hits[CATS[bl]] += 1
            r = rle(gm)
            annos.append({"id": aid, "image_id": imid, "category_id": 1, "segmentation": r,
                          "area": float(gm.sum()), "iscrowd": 0,
                          "bbox": [float(x) for x in maskutil.toBbox(r)]})
            aid += 1
        for pm, sc, lb in pl:
            dets.append({"image_id": imid, "category_id": 1, "segmentation": rle(pm), "score": sc})
        imid += 1

    # COVERAGE: did this GT object get any COCO-class detection at all?
    # Without this, a low mean IoU cannot be told apart from honest class
    # mismatch - many DAVIS objects are simply not COCO categories.
    seq_cov = {}
    for gid, v in seq_iou.items():
        v = np.array(v)
        seq_cov[gid] = {
            "mean_iou": round(float(v.mean()), 5),
            "frac_frames_iou_ge_0.5": round(float((v >= 0.5).mean()), 5),
            "frac_frames_any_overlap": round(float((v > 0.0).mean()), 5),
            "covered": bool(v.mean() >= COVER_TH),
        }
        OBJ_MEAN_IOU.append(v.mean())
        OBJ_COVERED.append(bool(v.mean() >= COVER_TH))
        OBJ_KEY.append((seq, int(gid)))
    per_seq[seq] = {"n_frames": len(GL), "n_instances": len(seq_iou),
                    "mean_iou": round(float(np.mean([np.mean(v) for v in seq_iou.values()])), 5),
                    "n_objects_covered": int(sum(1 for d in seq_cov.values() if d["covered"])),
                    "per_instance": seq_cov}

    # --- temporal ---
    s_jump = s_pairs = s_flick = s_fpairs = 0
    s_pred, s_gt = [], []
    for t in range(len(GL) - 1):
        la, lb_ = PL[t], PL[t + 1]
        assoc = {}
        if la and lb_:
            M = np.zeros((len(la), len(lb_)))
            for i in range(len(la)):
                for j in range(len(lb_)):
                    M[i, j] = iou(la[i][0], lb_[j][0])
            r_, c_ = linear_sum_assignment(-M)
            assoc = dict((int(i), int(j)) for i, j in zip(r_, c_) if M[i, j] >= LINK_TH)
        for gid, gm_a in GL[t].items():
            gm_b = GL[t + 1].get(gid)
            if gm_b is None:
                continue
            ia = max(range(len(la)), key=lambda i: iou(gm_a, la[i][0])) if la else -1
            ib = max(range(len(lb_)), key=lambda i: iou(gm_b, lb_[i][0])) if lb_ else -1
            va = iou(gm_a, la[ia][0]) if ia >= 0 else 0.0
            vb = iou(gm_b, lb_[ib][0]) if ib >= 0 else 0.0
            det_a, det_b = va >= DET_TH, vb >= DET_TH
            s_fpairs += 1
            flicker_pairs += 1
            if det_a != det_b:
                s_flick += 1
                flicker += 1
            if det_a and det_b:
                s_pairs += 1
                pairs_checked += 1
                if assoc.get(ia, -999) != ib:
                    s_jump += 1
                    jumps += 1
                pv = iou(la[ia][0], lb_[ib][0])
                gv = iou(gm_a, gm_b)
                s_pred.append(pv); s_gt.append(gv)
                pred_stab.append(pv); gt_stab.append(gv)
    per_seq_temporal[seq] = {
        "consecutive_pairs_both_detected": s_pairs,
        "identity_jump_rate": round(float(s_jump / s_pairs), 5) if s_pairs else None,
        "presence_flicker_rate": round(float(s_flick / s_fpairs), 5) if s_fpairs else None,
        "pred_frame_to_frame_iou": round(float(np.mean(s_pred)), 5) if s_pred else None,
        "gt_frame_to_frame_iou": round(float(np.mean(s_gt)), 5) if s_gt else None,
    }
    d = per_seq_temporal[seq]
    print("[%2d/%d] %-22s frames=%3d J=%.4f jump=%s flicker=%s pred_t2t=%s gt_t2t=%s  %.0fs"
          % (si + 1, len(SEQS), seq, len(GL), per_seq[seq]["mean_iou"], d["identity_jump_rate"],
             d["presence_flicker_rate"], d["pred_frame_to_frame_iou"], d["gt_frame_to_frame_iou"],
             time.time() - t0), flush=True)
    del GL, PL

print("\nframes scored:", n_scored_frames, "| GT instance-frames:", n_inst_frames,
      "| predicted masks:", n_pred_masks)
assert n_scored_frames >= 500, "only %d frames scored - nothing was measured" % n_scored_frames
assert n_inst_frames >= 500, "only %d GT instance-frames" % n_inst_frames
assert n_pred_masks > 0, "the model predicted no masks at all"

J_mean = float(np.mean([per_seq[s]["mean_iou"] for s in SEQS]))
J_instframe = float(np.mean(all_ious))
frac50 = float(np.mean(np.array(all_ious) >= 0.5))
OBJ_MEAN_IOU = np.array(OBJ_MEAN_IOU)
OBJ_COVERED = np.array(OBJ_COVERED)
n_objects = int(len(OBJ_MEAN_IOU))
coverage = float(OBJ_COVERED.mean())
J_matched = float(OBJ_MEAN_IOU[OBJ_COVERED].mean()) if OBJ_COVERED.any() else 0.0
J_allobj = float(OBJ_MEAN_IOU.mean())
frac_any = float(np.mean(np.array(all_ious) > 0.0))
print("GT objects (sequence, instance) tracked      = %d" % n_objects)
print("J mean (over sequences)                      = %.5f" % J_mean)
print("mean IoU over all GT instance-frames         = %.5f" % J_instframe)
print("mean IoU over all GT objects                 = %.5f" % J_allobj)
print("fraction of GT instance-frames with IoU>=0.5 = %.5f" % frac50)
print("fraction of GT instance-frames with ANY overlap = %.5f" % frac_any)
print("")
print("=== COVERAGE (object got any COCO-class detection, mean IoU >= %.2f) ===" % COVER_TH)
print("COVERAGE                                     = %.5f  (%d / %d objects)"
      % (coverage, int(OBJ_COVERED.sum()), n_objects))
print("MATCHED-OBJECT mean IoU (covered objects)    = %.5f" % J_matched)
print("uncovered objects, mean IoU                  = %.5f"
      % (float(OBJ_MEAN_IOU[~OBJ_COVERED].mean()) if (~OBJ_COVERED).any() else 0.0))
print("")
print("uncovered GT objects (class-mismatch cases, not segmentation failures):")
for k, (seq_, gid_) in enumerate(OBJ_KEY):
    if not OBJ_COVERED[k]:
        print("   %-22s instance %-3d mean IoU %.4f" % (seq_, gid_, OBJ_MEAN_IOU[k]))
assert n_objects >= 20, "only %d GT objects tracked" % n_objects

In [ ]:
# ---- class-agnostic mask AP ----
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
GTJ = "/kaggle/working/davis_gt.json"
DTJ = "/kaggle/working/davis_dt.json"
json.dump({"images": images, "annotations": annos,
           "categories": [{"id": 1, "name": "object"}]}, open(GTJ, "w"))
json.dump(dets, open(DTJ, "w"))
print("coco-format: images=%d gt_anns=%d dets=%d" % (len(images), len(annos), len(dets)))
assert len(annos) > 500 and len(dets) > 100

cg = COCO(GTJ)
cd = cg.loadRes(DTJ)
E = COCOeval(cg, cd, "segm")
E.evaluate(); E.accumulate(); E.summarize()
AP_NAMES = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
            "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]
mask_ap = dict((n, round(float(v), 5)) for n, v in zip(AP_NAMES, E.stats))
print("class-agnostic mask AP:", json.dumps(mask_ap, indent=1))
assert mask_ap["AP"] > 0.0, "mask AP is zero - the evaluation plumbing is broken"

In [ ]:
assert pairs_checked >= 200, "only %d consecutive pairs with the object detected in both frames" % pairs_checked
jump_rate = float(jumps / pairs_checked)
flick_rate = float(flicker / flicker_pairs)
pred_t2t = float(np.mean(pred_stab))
gt_t2t = float(np.mean(gt_stab))
print("=== TEMPORAL ===")
print("consecutive GT-object pairs examined:", flicker_pairs, "| both frames detected:", pairs_checked)
print("identity jump rate                  : %.5f" % jump_rate)
print("presence flicker rate               : %.5f" % flick_rate)
print("predicted mask frame-to-frame IoU   : %.5f" % pred_t2t)
print("GT mask frame-to-frame IoU (motion) : %.5f" % gt_t2t)
print("stability gap (gt - pred)           : %.5f" % (gt_t2t - pred_t2t))
print("COCO label assigned to well-matched DAVIS objects:", lab_hits.most_common(15))


# ---- BAR CHECK: judge this run against the published number for the same
# ---- model on the same benchmark. A number far below the bar is a bug
# ---- report about this pipeline, not a finding about the model.
BAR_PASS, BAR_REF = 0.60, "Mask R-CNN is COCO-trained and DAVIS objects are class-agnostic, so coverage is the number that makes matched IoU interpretable"
verdict = "PASS" if J_matched >= BAR_PASS else "BELOW BAR"
print("=" * 64)
print("BAR: matched-object mean IoU >= %.2f AND coverage reported" % BAR_PASS)
print("GOT: matched-object mean IoU  = %.5f" % J_matched)
print("     coverage                 = %.5f  (%d / %d GT objects)" % (coverage, int(OBJ_COVERED.sum()), n_objects))
print("     mean IoU over ALL objects= %.5f" % J_allobj)
print("VERDICT: %s   (%s)" % (verdict, BAR_REF))
print("=" * 64)

out = {
    "task": "video instance segmentation on DAVIS 2017, 480p",
    "model": "torchvision maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights.COCO_V1, as published, no fine-tuning, run independently per frame",
    "model_published_coco_metrics": WT.meta.get("_metrics"),
    "split": SPLIT_SRC,
    "n_sequences": len(SEQS),
    "sequences": SEQS,
    "n_frames_scored": n_scored_frames,
    "n_gt_instance_frames": n_inst_frames,
    "n_predicted_masks": n_pred_masks,
    "score_threshold": SCORE_TH,
    "mask_binarisation_threshold": MASK_TH,
    "matching": "class-agnostic; a GT instance is matched to the predicted mask of highest mask IoU regardless of COCO class",
    "per_frame_quality": {
        "J_mean_over_sequences": round(J_mean, 5),
        "mean_iou_over_all_gt_instance_frames": round(J_instframe, 5),
        "mean_iou_over_all_gt_objects": round(J_allobj, 5),
        "fraction_instance_frames_iou_ge_0.5": round(frac50, 5),
        "fraction_instance_frames_any_overlap": round(frac_any, 5),
        "mask_AP_class_agnostic": mask_ap,
    },
    "coverage": {
        "definition": "a GT object counts as COVERED if the mean over its frames of its best predicted-mask IoU reaches %.2f. Coverage separates honest COCO class mismatch (the object is not a COCO category, so nothing detects it) from poor segmentation of objects that WERE detected." % COVER_TH,
        "coverage_threshold_mean_iou": COVER_TH,
        "n_gt_objects": n_objects,
        "n_objects_covered": int(OBJ_COVERED.sum()),
        "coverage": round(coverage, 5),
        "matched_object_mean_iou": round(J_matched, 5),
        "uncovered_object_mean_iou": round(float(OBJ_MEAN_IOU[~OBJ_COVERED].mean()) if (~OBJ_COVERED).any() else 0.0, 5),
        "uncovered_objects": [{"sequence": OBJ_KEY[k][0], "instance": OBJ_KEY[k][1],
                               "mean_iou": round(float(OBJ_MEAN_IOU[k]), 5)}
                              for k in range(n_objects) if not OBJ_COVERED[k]],
    },
    "temporal_consistency": {
        "definition": {
            "identity_jump_rate": "over consecutive frame pairs where the GT object is detected (IoU>=0.5) in both frames, the fraction where the predicted mask covering it in frame t and the one covering it in frame t+1 are NOT linked to each other by Hungarian IoU matching at IoU>=0.5 - the object's mask identity jumped",
            "presence_flicker_rate": "fraction of consecutive GT-object pairs where the object is detected in one frame but not the other",
            "pred_frame_to_frame_iou": "mean IoU between the predicted masks covering the same GT object in frames t and t+1",
            "gt_frame_to_frame_iou": "the same quantity for the GT masks - the motion baseline, so the gap is model instability and not object movement",
        },
        "link_iou_threshold": LINK_TH,
        "detected_iou_threshold": DET_TH,
        "consecutive_gt_object_pairs": flicker_pairs,
        "pairs_detected_in_both_frames": pairs_checked,
        "identity_jump_rate": round(jump_rate, 5),
        "presence_flicker_rate": round(flick_rate, 5),
        "pred_frame_to_frame_iou": round(pred_t2t, 5),
        "gt_frame_to_frame_iou": round(gt_t2t, 5),
        "stability_gap_gt_minus_pred": round(gt_t2t - pred_t2t, 5),
    },
    "per_sequence": per_seq,
    "per_sequence_temporal": per_seq_temporal,
    "coco_labels_on_well_matched_davis_objects": dict(lab_hits.most_common(20)),
    "what_this_does_not_show": [
        "Mask R-CNN is not a video model and has no tracking component; these temporal numbers describe per-frame detection re-run independently, which is exactly the baseline a video method has to beat.",
        "DAVIS objects are class-agnostic while Mask R-CNN predicts COCO classes, so objects outside the 80 COCO classes can only be matched by accident. The label table shows what it actually assigned.",
        "One model on one dataset. Nothing here ranks this model against another.",
        "Coverage and matched-object IoU must be read together. Matched-object IoU alone flatters the model by excluding everything it never found; coverage alone says nothing about mask quality. The overall figures over all GT objects are reported too.",
        "J is computed on annotated frames of the chosen split at a fixed score threshold of 0.5; a different threshold moves both the IoU and the flicker rate.",
        "The identity jump rate is a property of this specific linking rule (Hungarian, IoU>=0.5) and would change under a different association threshold.",
    ],
}
json.dump(out, open("/kaggle/working/results.json", "w"), indent=1)
print(json.dumps(out, indent=1)[:5000])
print("\nWROTE /kaggle/working/results.json")

# The COCO-format dumps are 7 MB of RLE and only exist to feed COCOeval above.
# Leaving them in the output makes the result download break on a throttled link.
for _f in (GTJ, DTJ):
    try:
        os.remove(_f)
        print("removed intermediate", _f)
    except OSError as _e:
        print("could not remove", _f, _e)
print("final /kaggle/working:", sorted(os.listdir("/kaggle/working")))